# Maze Crawler — Submit *IL Agent (bunterr-trained)*

**Setup before running:**
1. Upload `weights.pt` (~1 MB) as a Kaggle Dataset and attach it here.
2. Create this notebook from the competition page so `crawl` is available.
3. **Run all** cells. Validation runs the 5 gates from the scope. Only submit if they pass.
4. **Save Version**, then submit `/kaggle/working/submission.tar.gz`.

This notebook also writes `v2.py` as a baseline opponent so we can measure IL vs v2 head-to-head.

In [1]:
# === Locate weights.pt ===
import os, shutil, glob
found = None
for p in glob.glob('/kaggle/input/datasets/alihamzapeenek/weights/weights.pt') + glob.glob('/kaggle/input/*/*/weights.pt') + ['/kaggle/working/weights.pt']:
    if os.path.exists(p): found = p; break
assert found is not None, 'weights.pt not found. Attach it as a dataset or upload to /kaggle/working/.'
shutil.copyfile(found, '/kaggle/working/weights.pt')
print('weights.pt copied from', found, '->', '/kaggle/working/weights.pt',
      'size', os.path.getsize('/kaggle/working/weights.pt'), 'bytes')

weights.pt copied from /kaggle/input/datasets/alihamzapeenek/weights/weights.pt -> /kaggle/working/weights.pt size 1008004 bytes


In [2]:
%%writefile main.py
"""IL agent — trained CNN policy that mimics bunterr.

At inference time, for each friendly unit we encode a 21x21x18 grid + 8 scalars,
batch all units, run one forward pass, take argmax over valid actions. CPU only.

`agent` is the last function (so the Kaggle loader picks it up). If torch or the
weights fail to load, we fall back to a v2-style heuristic so the bot never errors.
"""
import os, sys, traceback
import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    _TORCH_OK = True
except Exception:
    _TORCH_OK = False

# =============================================================
#  Encoder (identical to training-time encoder.py)
# =============================================================
GRID_HW = 21
HALF = GRID_HW // 2
NUM_CHANNELS = 18
NUM_SCALARS = 8
WIDTH = 20
FACTORY, SCOUT, WORKER, MINER = 0, 1, 2, 3

ACTIONS = (
    "IDLE",
    "NORTH", "SOUTH", "EAST", "WEST",
    "JUMP_NORTH", "JUMP_SOUTH", "JUMP_EAST", "JUMP_WEST",
    "BUILD_SCOUT_NORTH",  "BUILD_SCOUT_SOUTH",  "BUILD_SCOUT_EAST",  "BUILD_SCOUT_WEST",
    "BUILD_WORKER_NORTH", "BUILD_WORKER_SOUTH", "BUILD_WORKER_EAST", "BUILD_WORKER_WEST",
    "BUILD_MINER_NORTH",  "BUILD_MINER_SOUTH",  "BUILD_MINER_EAST",  "BUILD_MINER_WEST",
    "BUILD_NORTH", "BUILD_SOUTH", "BUILD_EAST", "BUILD_WEST",
    "REMOVE_NORTH", "REMOVE_SOUTH", "REMOVE_EAST", "REMOVE_WEST",
    "TRANSFORM",
    "TRANSFER_NORTH", "TRANSFER_SOUTH", "TRANSFER_EAST", "TRANSFER_WEST",
)
NUM_ACTIONS = len(ACTIONS)
ACT_TO_IDX = {a: i for i, a in enumerate(ACTIONS)}

_MOVES = ("IDLE", "NORTH", "SOUTH", "EAST", "WEST")
_TRANSFERS = ("TRANSFER_NORTH", "TRANSFER_SOUTH", "TRANSFER_EAST", "TRANSFER_WEST")
_VALID = {
    FACTORY: ("IDLE",
              "NORTH", "SOUTH", "EAST", "WEST",
              "JUMP_NORTH", "JUMP_SOUTH", "JUMP_EAST", "JUMP_WEST",
              "BUILD_SCOUT_NORTH",  "BUILD_SCOUT_SOUTH",  "BUILD_SCOUT_EAST",  "BUILD_SCOUT_WEST",
              "BUILD_WORKER_NORTH", "BUILD_WORKER_SOUTH", "BUILD_WORKER_EAST", "BUILD_WORKER_WEST",
              "BUILD_MINER_NORTH",  "BUILD_MINER_SOUTH",  "BUILD_MINER_EAST",  "BUILD_MINER_WEST",
              *_TRANSFERS),
    SCOUT:   (*_MOVES, *_TRANSFERS),
    WORKER:  (*_MOVES, "BUILD_NORTH","BUILD_SOUTH","BUILD_EAST","BUILD_WEST",
              "REMOVE_NORTH","REMOVE_SOUTH","REMOVE_EAST","REMOVE_WEST", *_TRANSFERS),
    MINER:   (*_MOVES, "TRANSFORM", *_TRANSFERS),
}

def _valid_mask(t):
    m = np.zeros(NUM_ACTIONS, dtype=bool)
    for a in _VALID.get(t, ("IDLE",)):
        m[ACT_TO_IDX[a]] = True
    return m

_MAX_E = (1.0, 100.0, 300.0, 500.0)

def _pk(k):
    a, b = k.split(",")
    return int(a), int(b)

def _obs_get(obs, name, default=None):
    """Works with both dict and kaggle_environments Struct."""
    try:
        v = obs[name] if isinstance(obs, dict) else getattr(obs, name)
        if v is not None: return v
    except Exception:
        pass
    return default

def encode(obs, unit_uid):
    robots = _obs_get(obs, "robots", {}) or {}
    unit = robots[unit_uid]
    utype, ucol, urow, uenergy, uowner = unit[0], unit[1], unit[2], unit[3], unit[4]
    move_cd = unit[5] if len(unit) > 5 else 0
    jump_cd = unit[6] if len(unit) > 6 else 0
    build_cd = unit[7] if len(unit) > 7 else 0
    south = _obs_get(obs, "southBound", 0)
    north = _obs_get(obs, "northBound", south + 19)
    walls = _obs_get(obs, "walls", []) or []
    step = _obs_get(obs, "step", 0)
    g = np.zeros((NUM_CHANNELS, GRID_HW, GRID_HW), dtype=np.float32)
    for gr in range(GRID_HW):
        for gc in range(GRID_HW):
            ac = ucol + (gc - HALF); ar = urow + (gr - HALF)
            if not (0 <= ac < WIDTH and south <= ar <= north): continue
            idx = (ar - south) * WIDTH + ac
            if 0 <= idx < len(walls):
                w = walls[idx]
                if w != -1:
                    g[4, gr, gc] = 1.0
                    if w & 1: g[0, gr, gc] = 1.0
                    if w & 2: g[1, gr, gc] = 1.0
                    if w & 4: g[2, gr, gc] = 1.0
                    if w & 8: g[3, gr, gc] = 1.0
    for ruid, r in robots.items():
        rt, rc, rr_, re_, ro = r[0], r[1], r[2], r[3], r[4]
        gc = rc - ucol + HALF; gr = rr_ - urow + HALF
        if not (0 <= gc < GRID_HW and 0 <= gr < GRID_HW): continue
        if ro == uowner:
            if rt == 0: g[5, gr, gc] = 1.0
            elif rt == 1: g[6, gr, gc] = 1.0
            elif rt == 2: g[7, gr, gc] = 1.0
            elif rt == 3: g[8, gr, gc] = 1.0
        else:
            g[9, gr, gc] = 1.0
        if rt == 0: g[14, gr, gc] = min(1.0, re_ / 1000.0)
        else:        g[14, gr, gc] = max(0.0, min(1.0, re_ / (_MAX_E[rt] or 1.0)))
        if ruid == unit_uid: g[15, gr, gc] = 1.0
    for k, m in (_obs_get(obs, "mines", {}) or {}).items():
        c, r = _pk(k); gc = c - ucol + HALF; gr = r - urow + HALF
        if 0 <= gc < GRID_HW and 0 <= gr < GRID_HW:
            owner = m[2] if len(m) > 2 else -1
            if owner == uowner: g[10, gr, gc] = 1.0
            else:                g[11, gr, gc] = 1.0
    for k in (_obs_get(obs, "miningNodes", {}) or {}):
        c, r = _pk(k); gc = c - ucol + HALF; gr = r - urow + HALF
        if 0 <= gc < GRID_HW and 0 <= gr < GRID_HW: g[12, gr, gc] = 1.0
    for k in (_obs_get(obs, "crystals", {}) or {}):
        c, r = _pk(k); gc = c - ucol + HALF; gr = r - urow + HALF
        if 0 <= gc < GRID_HW and 0 <= gr < GRID_HW: g[13, gr, gc] = 1.0
    g[16, :, :] = min(1.0, step / 500.0)
    si = 10.0 - 8.0 * min(1.0, step / 450.0)
    g[17, :, :] = 1.0 / max(si, 1.0)
    s = np.zeros(NUM_SCALARS, dtype=np.float32)
    s[0] = utype / 3.0
    s[1] = min(1.0, uenergy / 1000.0) if utype == 0 else min(1.0, uenergy / (_MAX_E[utype] or 1.0))
    s[2] = min(1.0, move_cd / 2.0)
    s[3] = min(1.0, jump_cd / 20.0) if utype == 0 else 0.0
    s[4] = min(1.0, build_cd / 10.0) if utype == 0 else 0.0
    s[5] = max(0.0, min(1.0, (urow - south) / 20.0))
    s[6] = min(1.0, step / 500.0)
    s[7] = float(uowner)
    return g, s, _valid_mask(utype)

# =============================================================
#  Model — same architecture as training
# =============================================================
if _TORCH_OK:
    class _ResBlock(nn.Module):
        def __init__(self, c):
            super().__init__()
            self.c1 = nn.Conv2d(c, c, 3, padding=1); self.b1 = nn.BatchNorm2d(c)
            self.c2 = nn.Conv2d(c, c, 3, padding=1); self.b2 = nn.BatchNorm2d(c)
        def forward(self, x):
            return F.relu(x + self.b2(self.c2(F.relu(self.b1(self.c1(x))))))

    class CrawlNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.stem = nn.Sequential(nn.Conv2d(NUM_CHANNELS, 64, 3, padding=1),
                                       nn.BatchNorm2d(64), nn.ReLU())
            self.blocks = nn.Sequential(_ResBlock(64), _ResBlock(64), _ResBlock(64))
            self.pool = nn.AdaptiveAvgPool2d(1)
            self.head = nn.Sequential(nn.Linear(64 + NUM_SCALARS, 128), nn.ReLU(),
                                       nn.Linear(128, NUM_ACTIONS))
        def forward(self, g, s, m):
            h = self.pool(self.blocks(self.stem(g))).flatten(1)
            z = torch.cat([h, s], dim=1)
            logits = self.head(z)
            return logits.masked_fill(~m, float('-inf'))

# =============================================================
#  Load weights once at import
# =============================================================
_MODEL = None
if _TORCH_OK:
    try:
        torch.set_num_threads(1)
        here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else "."
        for p in (os.path.join(here, "weights.pt"), "./weights.pt", "/kaggle/working/weights.pt"):
            if os.path.exists(p):
                ckpt = torch.load(p, map_location="cpu", weights_only=False)
                _MODEL = CrawlNet()
                _MODEL.load_state_dict(ckpt["model_state"])
                _MODEL.eval()
                break
    except Exception:
        _MODEL = None  # fall back to heuristic

# =============================================================
#  Safe fallback (heuristic) if the model isn't available
# =============================================================
def _fallback_agent(obs, config):
    me = _obs_get(obs, "player", 0)
    robots = _obs_get(obs, "robots", {}) or {}
    actions = {}
    for uid, d in robots.items():
        if d[4] != me: continue
        actions[uid] = "BUILD_WORKER" if d[0] == 0 else "NORTH"
    return actions

# =============================================================
#  Agent (MUST be the last function defined)
# =============================================================
def agent(obs, config):
    if _MODEL is None:
        return _fallback_agent(obs, config)
    try:
        me = _obs_get(obs, "player", 0)
        robots = _obs_get(obs, "robots", {}) or {}
        my = [uid for uid, d in robots.items() if d[4] == me]
        if not my:
            return {}
        gs, ss, ms = [], [], []
        for uid in my:
            g, s, m = encode(obs, uid)
            gs.append(g); ss.append(s); ms.append(m)
        Xg = torch.from_numpy(np.stack(gs))
        Xs = torch.from_numpy(np.stack(ss))
        Mk = torch.from_numpy(np.stack(ms))
        with torch.no_grad():
            pred = _MODEL(Xg, Xs, Mk).argmax(1).numpy()
        return {uid: ACTIONS[int(p)] for uid, p in zip(my, pred)}
    except Exception:
        return _fallback_agent(obs, config)


Writing main.py


In [3]:
%%writefile v2.py
"""
Maze Crawler - Harvester v2
===========================
v1's PROVEN survival logic (factory unchanged) + an economy layer on the units:
  * Scouts actively seek nearby crystals, then dump energy into the uncapped
    factory via TRANSFER when adjacent -> grows the factory's tiebreaker hoard.
  * Workers still clear walls / rescue the factory, and also bank spare energy.
The factory keeps marching north exactly as in v1 (do not regress survival).

`agent` MUST remain the LAST function. Types 0=F 1=S 2=W 3=M. Walls N1 E2 S4 W8.
robot = [type,col,row,energy,owner,move_cd,jump_cd,build_cd]
"""

from collections import deque

STATE = {"step": -1, "last_south": None}
DIRS = {"NORTH": (0, 1), "SOUTH": (0, -1), "EAST": (1, 0), "WEST": (-1, 0)}
DIRBIT = {"NORTH": 1, "EAST": 2, "SOUTH": 4, "WEST": 8}
TO_FACTORY = {(0, 1): "TRANSFER_NORTH", (0, -1): "TRANSFER_SOUTH",
              (1, 0): "TRANSFER_EAST", (-1, 0): "TRANSFER_WEST"}
FACTORY, SCOUT, WORKER, MINER = 0, 1, 2, 3
TARGET_SCOUTS = 2
TARGET_WORKERS = 2


def _cfg(config, name, default):
    try:
        v = getattr(config, name)
        if v is not None:
            return v
    except Exception:
        pass
    try:
        return config[name]
    except Exception:
        return default


def _obs(obs, name, default):
    try:
        v = getattr(obs, name)
        if v is not None:
            return v
    except Exception:
        pass
    try:
        return obs[name]
    except Exception:
        return default


def _bfs_first_step(start, can_step):
    visited = {start}
    firstmove = {start: None}
    q = deque([start])
    best, best_row = start, start[1]
    while q:
        c, r = q.popleft()
        if r > best_row:
            best_row, best = r, (c, r)
        for dirn, (dx, dy) in DIRS.items():
            if not can_step(c, r, dirn):
                continue
            nxt = (c + dx, r + dy)
            if nxt in visited:
                continue
            visited.add(nxt)
            firstmove[nxt] = firstmove[(c, r)] if firstmove[(c, r)] is not None else dirn
            q.append(nxt)
    if best_row <= start[1]:
        return None
    return firstmove.get(best)


def _decide(obs, config):
    W = int(_cfg(config, "width", 20))
    walls = list(_obs(obs, "walls", []))
    south = int(_obs(obs, "southBound", 0))
    north = int(_obs(obs, "northBound", south + (len(walls) // W if W else 20) - 1))
    me = _obs(obs, "player", 0)
    robots = _obs(obs, "robots", {})
    crystals = _obs(obs, "crystals", {}) or {}
    episode_steps = int(_cfg(config, "episodeSteps", 501))
    scout_cost = int(_cfg(config, "scoutCost", 50))
    worker_cost = int(_cfg(config, "workerCost", 200))
    remove_cost = int(_cfg(config, "wallRemoveCost", 100))

    last_south = STATE.get("last_south")
    if last_south is not None and south < last_south - 2:
        STATE["step"] = -1
    STATE["step"] += 1
    STATE["last_south"] = south
    step = STATE["step"]
    turns_left = max(20, episode_steps - step)

    def wbits(col, row):
        if not (0 <= col < W):
            return None
        i = (row - south) * W + col
        if 0 <= i < len(walls):
            v = walls[i]
            return None if v == -1 else v
        return None

    def on_board(col, row):
        return 0 <= col < W and south <= row <= north

    def can_step(col, row, dirn):
        b = wbits(col, row)
        if b is None:
            return False
        if b & DIRBIT[dirn]:
            return False
        dx, dy = DIRS[dirn]
        return on_board(col + dx, row + dy)

    my_units = {}
    enemy_cells, enemy_factory_cells = set(), set()
    for uid, d in robots.items():
        if d[4] == me:
            my_units[uid] = d
        else:
            enemy_cells.add((d[1], d[2]))
            if d[0] == FACTORY:
                enemy_factory_cells.add((d[1], d[2]))

    factory_uid = factory = None
    scouts, workers, miners = [], [], []
    for uid, d in my_units.items():
        if d[0] == FACTORY:
            factory_uid, factory = uid, d
        elif d[0] == SCOUT:
            scouts.append((uid, d))
        elif d[0] == WORKER:
            workers.append((uid, d))
        elif d[0] == MINER:
            miners.append((uid, d))

    actions, claimed, friendly_final = {}, set(), set()
    factory_pos = (factory[1], factory[2]) if factory else None
    factory_energy = factory[3] if factory else 0
    fcol = factory_pos[0] if factory_pos else W // 2

    def crystal_at(col, row):
        return ("%d,%d" % (col, row)) in crystals

    def adj_factory_dir(col, row):
        if factory_pos is None:
            return None
        return TO_FACTORY.get((factory_pos[0] - col, factory_pos[1] - row))

    def nearest_crystal(col, row, max_dist=8):
        best, bestscore = None, None
        for key, e in crystals.items():
            try:
                cc, cr = key.split(","); cc, cr = int(cc), int(cr)
            except Exception:
                continue
            if cr < south + 2:
                continue
            d = abs(cc - col) + abs(cr - row)
            if d == 0 or d > max_dist:
                continue
            sc = d - e * 0.03
            if bestscore is None or sc < bestscore:
                bestscore, best = sc, (cc, cr)
        return best

    def toward(col, row, target, fallback):
        pref = []
        if target:
            tc, tr = target
            if tr > row: pref.append("NORTH")
            if tc > col: pref.append("EAST")
            if tc < col: pref.append("WEST")
            if tr < row: pref.append("SOUTH")
        for d in fallback:
            if d not in pref:
                pref.append(d)
        return pref

    def pick_move(col, row, prefer, avoid_enemy=True, allow_south=False, col_bias=None):
        order, seen = [], set()
        for d in prefer:
            if d not in seen and (allow_south or d != "SOUTH"):
                seen.add(d); order.append(d)
        best, best_score = None, None
        for dirn in order:
            if not can_step(col, row, dirn):
                continue
            dx, dy = DIRS[dirn]
            nc, nr = col + dx, row + dy
            cell = (nc, nr)
            if cell in claimed:
                continue
            if avoid_enemy and cell in enemy_cells:
                continue
            score = order.index(dirn)
            if crystal_at(nc, nr):
                score -= 100
            if col_bias is not None:
                score += abs(nc - col_bias) * 0.01
            if best_score is None or score < best_score:
                best_score, best = score, (dirn, cell)
        return best if best else ("IDLE", (col, row))

    # ---------- SCOUTS: harvest crystals, bank into factory ----------
    for i, (uid, d) in enumerate(scouts):
        col, row, energy = d[1], d[2], d[3]
        tdir = adj_factory_dir(col, row)
        if tdir and energy >= 40:
            actions[uid] = tdir
            friendly_final.add((col, row)); claimed.add((col, row)); continue
        if energy <= 0:
            friendly_final.add((col, row)); claimed.add((col, row)); continue
        if i % 2 == 0:
            npref, bias = ["NORTH", "EAST", "WEST"], min(W - 1, col + 4)
        else:
            npref, bias = ["NORTH", "WEST", "EAST"], max(0, col - 4)
        tgt = nearest_crystal(col, row, 8)
        allow_s = tgt is not None and tgt[1] >= south + 2 and tgt[1] < row
        pref = toward(col, row, tgt, npref)
        act, cell = pick_move(col, row, pref, allow_south=allow_s, col_bias=bias)
        actions[uid] = act
        friendly_final.add(cell); claimed.add(cell)

    # ---------- WORKERS: clear walls / rescue factory / harvest ----------
    for uid, d in workers:
        col, row, energy = d[1], d[2], d[3]
        b = wbits(col, row)
        buf = row - south
        # rescue: feed a starving factory
        if factory_pos is not None and factory_energy < 150 and energy > 20:
            tdir = adj_factory_dir(col, row)
            if tdir:
                actions[uid] = tdir
                friendly_final.add((col, row)); claimed.add((col, row)); continue
        # clear a north wall ahead (helps the factory's lane)
        if (b is not None) and (b & 1) and energy > remove_cost + 5 and row < north and buf >= 1:
            actions[uid] = "REMOVE_NORTH"
            friendly_final.add((col, row)); claimed.add((col, row)); continue
        # bank spare energy when adjacent
        tdir = adj_factory_dir(col, row)
        if tdir and energy >= 120:
            actions[uid] = tdir
            friendly_final.add((col, row)); claimed.add((col, row)); continue
        tgt = nearest_crystal(col, row, 5)
        pref = toward(col, row, tgt, ["NORTH", "EAST", "WEST"])
        act, cell = pick_move(col, row, pref, col_bias=fcol)
        actions[uid] = act
        friendly_final.add(cell); claimed.add(cell)

    for uid, d in miners:
        col, row = d[1], d[2]
        act, cell = pick_move(col, row, ["NORTH", "EAST", "WEST"], col_bias=fcol)
        actions[uid] = act
        friendly_final.add(cell); claimed.add(cell)

    # ---------- FACTORY: identical to v1 (proven survival) ----------
    if factory is not None:
        col, row, energy = factory[1], factory[2], factory[3]
        jump_cd = factory[6] if len(factory) > 6 else 0
        build_cd = factory[7] if len(factory) > 7 else 0
        fb = wbits(col, row)
        north_wall = (fb is not None) and (fb & 1)
        buffer = row - south
        early_econ = step < 130

        def north_cell_safe():
            nc = (col, row + 1)
            if nc in enemy_factory_cells:
                return False
            if nc in friendly_final:
                return False
            return True

        def jump_ok():
            return jump_cd == 0 and (row + 2) <= north

        n_scouts = len(scouts)
        n_workers = len(workers)
        spawn_free = (not north_wall) and (row + 1 <= north) \
            and ((col, row + 1) not in friendly_final) \
            and ((col, row + 1) not in enemy_cells)
        build_action = None
        if build_cd == 0 and spawn_free and early_econ and buffer >= 3:
            if n_scouts < TARGET_SCOUTS and (energy - scout_cost) > (turns_left + 60):
                build_action = "BUILD_SCOUT"
            elif n_workers < TARGET_WORKERS and (energy - worker_cost) > (turns_left + 60):
                build_action = "BUILD_WORKER"

        def safe(cell):
            return cell not in friendly_final and cell not in enemy_factory_cells \
                and cell not in enemy_cells

        def northward_move():
            jr = jump_ok()
            # Jump to bypass a blocking wall, or proactively when buffer is thin
            # or in the fast-scroll endgame; otherwise SAVE the jump for walls.
            if jr and (north_wall or buffer <= 8 or step > 430):
                return "JUMP_NORTH"
            if not north_wall and (row + 1 <= north) and north_cell_safe() \
               and (col, row + 1) not in enemy_cells:
                return "NORTH"
            if jr:  # north blocked, jump is our bypass
                return "JUMP_NORTH"
            # route around using BFS, but NEVER step south (toward the boundary)
            fm = _bfs_first_step((col, row), can_step)
            if fm is not None and fm != "SOUTH":
                dx, dy = DIRS[fm]
                if safe((col + dx, row + dy)):
                    return fm
            for dirn in ("EAST", "WEST"):
                if can_step(col, row, dirn):
                    dx, dy = DIRS[dirn]
                    cell = (col + dx, row + dy)
                    if safe(cell) and cell not in claimed:
                        return dirn
            return "IDLE"

        actions[factory_uid] = build_action if build_action is not None else northward_move()

    return actions


def agent(obs, config):
    try:
        return _decide(obs, config)
    except Exception:
        actions = {}
        try:
            me = _obs(obs, "player", 0)
            for uid, d in _obs(obs, "robots", {}).items():
                if d[4] == me:
                    actions[uid] = "BUILD_WORKER" if d[0] == FACTORY else "NORTH"
        except Exception:
            pass
        return actions


Writing v2.py


In [4]:
# === GATE 1: model loads, mirror match runs without errors ===
from kaggle_environments import make
import time
env = make('crawl', configuration={'randomSeed': 42}, debug=True)
t0 = time.time()
env.run(['main.py', 'main.py'])
dt = time.time() - t0
last = env.steps[-1]; statuses = [s.status for s in last]
print('episode length:', len(env.steps), 'rewards:', [s.reward for s in last], 'statuses:', statuses)
print(f'mirror match runtime: {dt:.1f}s  (~{dt*1000/len(env.steps):.0f} ms/turn for both agents combined)')
assert 'ERROR' not in statuses and 'INVALID' not in statuses, 'GATE 1 FAILED: agent errored'
print('GATE 1 PASS: no errors.')

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 24.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game_arena
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environment

In [5]:
# === GATE 2: beats `random` >= 90% over 20 games ===
wins = 0; n = 20
for seed in range(n):
    env = make('crawl', configuration={'randomSeed': seed})
    env.run(['main.py', 'random'])
    r = env.steps[-1]
    if r[0].reward is not None and r[1].reward is not None and r[0].reward > r[1].reward:
        wins += 1
print(f'IL vs random (P0): {wins}/{n} wins')
wins2 = 0
for seed in range(n):
    env = make('crawl', configuration={'randomSeed': 100 + seed})
    env.run(['random', 'main.py'])
    r = env.steps[-1]
    if r[0].reward is not None and r[1].reward is not None and r[1].reward > r[0].reward:
        wins2 += 1
print(f'IL vs random (P1): {wins2}/{n} wins')
total = wins + wins2
print(f'GATE 2: {total}/{2*n} = {100*total/(2*n):.0f}%  -- need >= 90% ({int(0.9*2*n)})')
assert total >= int(0.9 * 2 * n), 'GATE 2 FAILED: IL did not beat random reliably'

IL vs random (P0): 0/20 wins
IL vs random (P1): 0/20 wins
GATE 2: 0/40 = 0%  -- need >= 90% (36)


AssertionError: GATE 2 FAILED: IL did not beat random reliably

In [6]:
# === GATE 3: beats v2 >= 60% over 20 games ===
wins = 0; n = 10
for seed in range(n):
    env = make('crawl', configuration={'randomSeed': seed})
    env.run(['main.py', 'v2.py'])
    r = env.steps[-1]
    if r[0].reward is not None and r[1].reward is not None and r[0].reward > r[1].reward:
        wins += 1
print(f'IL vs v2 (P0): {wins}/{n}')
wins2 = 0
for seed in range(n):
    env = make('crawl', configuration={'randomSeed': 200 + seed})
    env.run(['v2.py', 'main.py'])
    r = env.steps[-1]
    if r[0].reward is not None and r[1].reward is not None and r[1].reward > r[0].reward:
        wins2 += 1
print(f'IL vs v2 (P1): {wins2}/{n}')
total = wins + wins2
print(f'GATE 3: {total}/{2*n} = {100*total/(2*n):.0f}%  -- need >= 60% ({int(0.6*2*n)})')
if total < int(0.6 * 2 * n):
    print('GATE 3 SOFT-FAIL: IL did not clearly beat v2 yet.')
    print('   If it tied or just-lost, you can still consider submitting (will replace v1).')
    print('   If it lost badly, hold and we diagnose before submitting.')

IL vs v2 (P0): 0/10
IL vs v2 (P1): 0/10
GATE 3: 0/20 = 0%  -- need >= 60% (12)
GATE 3 SOFT-FAIL: IL did not clearly beat v2 yet.
   If it tied or just-lost, you can still consider submitting (will replace v1).
   If it lost badly, hold and we diagnose before submitting.


In [7]:
# === GATE 4: reach step 500 in >= 50% of mirror matches ===
reached = 0; n = 10
for seed in range(n):
    env = make('crawl', configuration={'randomSeed': 300 + seed})
    env.run(['main.py', 'main.py'])
    if len(env.steps) >= 500: reached += 1
print(f'GATE 4: {reached}/{n} mirror matches reached step 500')
if reached < n // 2:
    print('GATE 4 SOFT-FAIL: model gets caught at the boundary too often.')

GATE 4: 0/10 mirror matches reached step 500
GATE 4 SOFT-FAIL: model gets caught at the boundary too often.


In [8]:
# === GATE 5: inference time per turn under 200 ms ===
env = make('crawl', configuration={'randomSeed': 7})
import time
t0 = time.time(); env.run(['main.py', 'random']); dt = time.time() - t0
ms_per_turn = dt * 1000 / max(1, len(env.steps))
print(f'avg ms per turn (combined IL + random): {ms_per_turn:.1f}')
print(f'GATE 5: need < 200 ms per turn for our side  -- we are likely well below')

avg ms per turn (combined IL + random): 12.3
GATE 5: need < 200 ms per turn for our side  -- we are likely well below


In [ ]:
# === Render a sample game (visual sanity) ===
env = make('crawl', configuration={'randomSeed': 1})
env.run(['main.py', 'v2.py'])
env.render(mode='ipython', width=720, height=720)

In [ ]:
# === Package submission.tar.gz with main.py + weights.pt ===
import tarfile
with tarfile.open('/kaggle/working/submission.tar.gz', 'w:gz') as t:
    t.add('main.py', arcname='main.py')
    t.add('/kaggle/working/weights.pt', arcname='weights.pt')
import os; print('submission.tar.gz size:', os.path.getsize('/kaggle/working/submission.tar.gz'), 'bytes')
print('SAVE VERSION the notebook, then submit /kaggle/working/submission.tar.gz on the competition page.')